# 基於截斷策略的機器閱讀理解（MRC）

**學習目標**

1. 理解抽取式問答（Extractive QA）的核心任務：預測答案在 context 中的 start/end token 位置。
2. 掌握 offset_mapping 的作用，以及如何將字元級答案位置轉換為 token 級位置。
3. 學會用 `datasets.map(batched=True)` 高效預處理中文 MRC 資料集 CMRC-2018。
4. 使用 2026 標準的 `Trainer` + `TrainingArguments`（bf16、cosine、AdamW fused）訓練問答模型。
5. 用 `pipeline(device_map='auto')` 做端到端推論，並理解 safetensors 的安全與效能優勢。

**前置知識**

- 熟悉 BERT 的 `[CLS]` / `[SEP]` / `token_type_ids` 結構
- 了解 `AutoTokenizer` 的基本用法
- 建議先閱讀：`../02-named_entity_recognition/ner_demo.ipynb`（同樣是 token-level 標籤任務）

**與相鄰 notebook 的銜接**

- 上一個任務：`../02-named_entity_recognition/ner_demo.ipynb` — NER 預測每個 token 的類別
- 本任務：MRC 預測答案的起始/結束 token，屬於同一類「span prediction」範疇
- 下一個任務：`./mrc_sliding_window.ipynb` — 處理超長 context 的滑動視窗策略（本 notebook 用截斷；滑動視窗更精確）

In [ ]:
# Install / lock versions — run once, then restart kernel
# Pin to 2026-stable versions to ensure reproducibility
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "evaluate>=0.4" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "torch>=2.4"

## Step 1 匯入套件

`DataCollatorWithPadding` 在 batch 內動態補齊序列長度，避免浪費計算在過多的 padding token 上。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
    set_seed,
)
from datasets import load_dataset
import evaluate

# Fix global random seed for reproducibility
set_seed(42)

## Step 2 資料集下載

使用 `load_dataset("cmrc2018")` 直接從 HF Hub 拉取 CMRC-2018，這是目前最主流的中文抽取式閱讀理解基準。

資料集結構：
- `context`：一段中文段落
- `question`：針對段落提出的問題
- `answers`：`{text: [...], answer_start: [...]}`，答案在 context 中的字元級起始位置

In [ ]:
# Load CMRC-2018 directly from HF Hub
# No local path or Google Drive mount needed
raw_datasets = load_dataset("cmrc2018")
print(raw_datasets)

In [ ]:
# Inspect a training sample
raw_datasets["train"][24]

## Step 3 資料預處理

### 3.1 載入 Tokenizer

`hfl/chinese-macbert-base` 是 Roberta 架構的繁簡中文預訓練模型，適合 QA 任務。

In [ ]:
MODEL_ID = "hfl/chinese-macbert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(tokenizer)

### 3.2 理解 offset_mapping 與答案位置映射

抽取式 QA 的關鍵挑戰：CMRC-2018 的答案是**字元級**位置，但模型需要**token 級**的 start/end。

`return_offsets_mapping=True` 讓 tokenizer 額外回傳每個 token 對應到原始字串的字元範圍 `(char_start, char_end)`，利用這個映射就能從字元位置轉換到 token 位置。

**截斷策略說明**
- `truncation="only_second"`：只截斷 context（第二個序列），question 不截斷
- `max_length=384`：對 BERT 系列是常見的安全值（512 也可，但訓練稍慢）
- 截斷的代價：若答案剛好在截斷後的部分，會標記為「答案不在 context 內」（start=0, end=0）
- 更精確的做法是滑動視窗（見 `mrc_sliding_window.ipynb`）

In [ ]:
# Demonstrate on a small sample to understand offset_mapping
sample_dataset = raw_datasets["train"].select(range(5))

tokenized_sample = tokenizer(
    text=sample_dataset["question"],
    text_pair=sample_dataset["context"],
    return_offsets_mapping=True,
    max_length=384,
    truncation="only_second",
    padding=False,  # no padding in exploration; padding is handled by DataCollatorWithPadding
)

print("Keys:", tokenized_sample.keys())
print("Sequence length (sample 0):", len(tokenized_sample["input_ids"][0]))

In [ ]:
# token_type_ids: 0 = question tokens, 1 = context tokens
# This is how BERT distinguishes the two sequences
print("(token_id, token_type_id) for sample 0:")
print(list(zip(tokenized_sample["input_ids"][0], tokenized_sample["token_type_ids"][0]))[:30], "...")

In [ ]:
# offset_mapping: each entry is (char_start, char_end) in the original string
# Special tokens ([CLS], [SEP]) have (0, 0)
print("offset_mapping (sample 0):")
print(tokenized_sample["offset_mapping"][0][:30], "...")

### 3.3 答案位置轉換邏輯（核心算法）

以下逐步拆解「字元位置 → token 位置」的轉換過程：

1. 用 `sequence_ids()` 找出 context 在 token 序列中的起止邊界
2. 若答案的字元範圍完全超出截斷後的 context，標記 `start=0, end=0`（表示無答案）
3. 否則，從 context 兩端向內逼近，找到最精確的 start/end token

In [ ]:
# Walk through the mapping logic for one example
offset_mapping = tokenized_sample["offset_mapping"]

for idx in range(len(sample_dataset)):
    offset = offset_mapping[idx]
    answer = sample_dataset[idx]["answers"]
    start_char = answer["answer_start"][0]
    end_char = start_char + len(answer["text"][0])

    # Find context token boundaries using sequence_ids
    # sequence_ids returns: 0 for question tokens, 1 for context tokens, None for special tokens
    seq_ids = tokenized_sample.sequence_ids(idx)
    context_start = seq_ids.index(1)                          # first context token
    context_end = seq_ids.index(None, context_start) - 1     # last context token

    # Check if answer is outside the (possibly truncated) context
    if offset[context_end][1] < start_char or offset[context_start][0] > end_char:
        start_token_pos = 0
        end_token_pos = 0
    else:
        # Approach from left: find start token
        token_id = context_start
        while token_id <= context_end and offset[token_id][0] < start_char:
            token_id += 1
        start_token_pos = token_id

        # Approach from right: find end token
        token_id = context_end
        while token_id >= context_start and offset[token_id][1] > end_char:
            token_id -= 1
        end_token_pos = token_id

    decoded_answer = tokenizer.decode(
        tokenized_sample["input_ids"][idx][start_token_pos : end_token_pos + 1]
    )
    print(f"[{idx}] gold='{answer['text'][0]}' | char={start_char}~{end_char} "
          f"| token={start_token_pos}~{end_token_pos} | decoded='{decoded_answer}'")

### 3.4 封裝成 process_func 並批次處理

將上述邏輯封裝後，用 `dataset.map(batched=True, num_proc=4)` 並行處理。

> **為何 batched=True 快 3-5 倍？**
> - 非 batched 模式：對每筆樣本分別呼叫 Python 函數，Python overhead 大
> - batched=True：每次傳入一個 batch（預設 1000 筆），tokenizer 可向量化處理，I/O 也大幅減少

> **注意**：`max_length=384, padding=False` — 這裡不做 padding；padding 延後到 `DataCollatorWithPadding` 在 batch 內動態處理，能大幅減少 padding token 的比例。

In [ ]:
def process_func(examples):
    """Tokenize QA examples and map char-level answer positions to token-level.

    Args:
        examples: batch of {question, context, answers} from CMRC-2018

    Returns:
        tokenized batch with added start_positions and end_positions
    """
    tokenized = tokenizer(
        text=examples["question"],
        text_pair=examples["context"],
        return_offsets_mapping=True,
        max_length=384,
        truncation="only_second",
        padding=False,  # dynamic padding will be applied by DataCollatorWithPadding
    )
    offset_mapping = tokenized.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for idx, offset in enumerate(offset_mapping):
        answer = examples["answers"][idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        seq_ids = tokenized.sequence_ids(idx)
        context_start = seq_ids.index(1)
        context_end = seq_ids.index(None, context_start) - 1

        # Answer is outside the (truncated) context window
        if offset[context_end][1] < start_char or offset[context_start][0] > end_char:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Left approach: find start token
        token_id = context_start
        while token_id <= context_end and offset[token_id][0] < start_char:
            token_id += 1
        start_positions.append(token_id)

        # Right approach: find end token
        token_id = context_end
        while token_id >= context_start and offset[token_id][1] > end_char:
            token_id -= 1
        end_positions.append(token_id)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [ ]:
# Apply process_func across all splits in parallel
# batched=True + num_proc=4 speeds up tokenization by 3-5x compared to row-by-row
tokenized_datasets = raw_datasets.map(
    process_func,
    batched=True,
    num_proc=4,
    remove_columns=raw_datasets["train"].column_names,
)
print(tokenized_datasets)

## Step 4 載入模型

### 2026 標準載入方式

| 參數 | 說明 |
|---|---|
| `device_map='auto'` | 自動分配到 GPU；若 VRAM 不足可 offload 到 CPU/disk |
| `torch_dtype=torch.bfloat16` | bf16 相比 fp16 有更大的指數範圍，不需要 loss scaling，訓練更穩定；比 fp32 省一半記憶體 |
| `use_safetensors=True` | safetensors 格式不使用 Python pickle，載入更快且無任意代碼執行風險 |

> **VRAM 估算（macbert-base ~400 MB）**
> - fp32: ~1.6 GB；bf16: ~0.8 GB；足以在消費級 GPU 上訓練
> - 若只有 CPU，移除 `device_map='auto'` 或設為 `device_map='cpu'`

In [ ]:
# Load model with 2026 standard: device_map + bfloat16 + safetensors
# macbert-base is ~400 MB in bf16; runs on any GPU with >= 4 GB VRAM
model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
print(model.config.model_type)
print("Device map:", model.hf_device_map)

## Step 5 設定 TrainingArguments

### 2026 完整設定說明

| 參數 | 值 | 原因 |
|---|---|---|
| `bf16=True` | True | 與模型載入精度一致；比 fp16 訓練更穩定 |
| `optim='adamw_torch_fused'` | fused AdamW | PyTorch 2.x 的融合算子，比標準 AdamW 快約 10-15% |
| `warmup_ratio=0.1` | 0.1 | 前 10% steps 線性暖身，避免初期 loss spike |
| `lr_scheduler_type='cosine'` | cosine | 平滑衰減，收斂效果優於 linear |
| `max_grad_norm=1.0` | 1.0 | 梯度裁剪，防止梯度爆炸 |
| `eval_strategy='steps'` | steps | 比 epoch 更頻繁地監控，及早發現 overfitting |
| `save_safetensors=True` | True | 儲存為 safetensors 格式，更安全更快 |
| `load_best_model_at_end=True` | True | 訓練結束自動載入最優 checkpoint |
| `seed=42` | 42 | 固定訓練隨機性，確保可重現 |

> **Effective batch size** = `per_device_train_batch_size × gradient_accumulation_steps × GPU 數量`
> 本設定：32 × 1 × 1 = 32，與原始設定等效。
> 若 GPU VRAM 不足，可改 `per_device_train_batch_size=8, gradient_accumulation_steps=4` 達到同樣 effective batch size。

In [ ]:
# 2026 standard TrainingArguments
training_args = TrainingArguments(
    output_dir="models_for_qa",
    # batch & accumulation
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,   # increase if VRAM is limited
    # precision
    bf16=True,
    # optimizer & scheduler
    optim="adamw_torch_fused",       # PyTorch 2.x fused AdamW
    learning_rate=3e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    # training length
    num_train_epochs=3,
    # evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # logging
    logging_steps=50,
    # serialization
    save_safetensors=True,
    # reproducibility
    seed=42,
)

## Step 6 設定 Trainer

`DataCollatorWithPadding` 在每個 batch 內動態 padding 到該 batch 的最長序列，而非全部 padding 到 max_length=384。這在序列長度分佈不均時特別有效率。

> **注意**：`AutoModelForQuestionAnswering` 的 loss 是模型內建的交叉熵（對 start/end logits 各算一次），`Trainer` 會自動使用，不需要自訂 `compute_loss`。
> 若需計算 Exact Match / F1 指標，可參考 `mrc_sliding_window.ipynb` 的完整 `compute_metrics` 實作。

In [ ]:
# DataCollatorWithPadding: dynamic padding within each batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,  # 2026: replaces deprecated tokenizer= kwarg
)

## Step 7 模型訓練

訓練過程中 `Trainer` 會：
1. 每 50 steps 記錄 loss
2. 每 200 steps 評估 validation loss 並儲存 checkpoint（safetensors 格式）
3. 訓練結束後自動載入 eval_loss 最低的 checkpoint

In [ ]:
# Start training
# Expected training time on RTX 3090: ~8 min/epoch for CMRC-2018 train set (~10k samples)
trainer.train()

## Step 8 模型推論

### 2026 pipeline 載入方式

`pipeline(..., device_map='auto')` 自動偵測可用裝置；在沒有 GPU 的環境也能跑（使用 CPU），與 `from_pretrained(device_map='auto')` 語意一致，降低認知負擔。

In [ ]:
# Load the best checkpoint for inference
qa_pipeline = pipeline(
    "question-answering",
    model=trainer.model,
    tokenizer=tokenizer,
    device_map="auto",
)
print(qa_pipeline)

In [ ]:
# Simple end-to-end inference
result = qa_pipeline(
    question="小明在哪裡上班？",
    context="小明在北京上班。"
)
print(result)

In [ ]:
# Test with a CMRC-style example
example = raw_datasets["validation"][0]
result = qa_pipeline(
    question=example["question"],
    context=example["context"]
)
print("Question:", example["question"])
print("Gold answer:", example["answers"]["text"][0])
print("Predicted:", result["answer"], f"(score={result['score']:.4f})")

## Step 9 儲存模型（選用）

若要將訓練好的模型推送到 HF Hub 或本地保存，使用 `save_safetensors=True` 確保以安全格式序列化。

In [ ]:
# Save locally in safetensors format
SAVE_DIR = "models_for_qa/best"
trainer.model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")

# Optional: push to HF Hub
# Requires: huggingface-cli login
# trainer.push_to_hub(
#     "your-username/chinese-macbert-base-cmrc2018",
#     language="zh",
#     license="apache-2.0",
#     tags=["question-answering", "chinese", "cmrc2018"],
# )

## 小結

### 本 notebook 完成的事

1. **offset_mapping 映射**：用 `return_offsets_mapping=True` 將字元級答案位置轉換為 token 級 start/end position，這是抽取式 QA 最核心的預處理步驟。

2. **截斷策略**：`truncation="only_second"` 只截斷 context，保留完整 question。當答案被截斷時標記為 (0, 0)（無答案）。更完善的做法是滑動視窗（見 `mrc_sliding_window.ipynb`）。

3. **2026 標準載入**：`device_map='auto' + torch_dtype=bfloat16 + use_safetensors=True` 提供自動裝置分配、穩定的半精度訓練以及安全的模型序列化。

4. **現代化 TrainingArguments**：補齊 `bf16`、`optim='adamw_torch_fused'`、`warmup_ratio`、`cosine scheduler`、`eval_strategy='steps'`，收斂品質優於原版。

5. **動態 padding**：`DataCollatorWithPadding` 在 batch 內動態對齊序列長度，省下無謂的 padding 計算。

### 練習題

1. **截斷率分析**：在 `process_func` 中加入計數器，統計有多少比例的樣本答案被截斷（start=0, end=0）。這個比例高嗎？對模型訓練有何影響？

2. **加入 Exact Match / F1 評測**：使用 `evaluate.load('squad')` 計算 Exact Match 和 F1 指標，並接入 `Trainer` 的 `compute_metrics`。（提示：QA 的 compute_metrics 需要解碼 logits，參考官方 QA example。）

3. **滑動視窗對比**：修改 `process_func`，加入 `stride` 參數，實作滑動視窗策略（`truncation='only_second', stride=128`）。比較截斷率與最終 F1 的差異。

4. **不同 max_length 的效能權衡**：分別嘗試 `max_length=256` 和 `max_length=512`，比較訓練速度、截斷率與 validation F1。